Maybe try to read in the opera datasets into MintPy for improved time series velocities and ascending/descending decomposition? Then clip the datasets by the aquifers for zonal statistics?

In [17]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

from mintpy import view, tsview
import os
from pathlib import Path
import matplotlib.pyplot as plt
import geopandas as gpd

from mintpy.cli.tsview import cmd_line_parse
from mintpy.tsview import timeseriesViewer
from mintpy.view import viewer
from mintpy import geocode
from mintpy import info
import h5py
import numpy as np
from mintpy.utils.utils0 import calc_azimuth_from_east_north_obs, azimuth2heading_angle
import rasterio as rio

import rioxarray as rxr
import xarray as xr

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
gdf = gpd.read_file(Path(os.getcwd()).parent.parent / "raw_data/TBA_full.shp")


data_storage = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data"    # linux
# data_storage = Path('/mnt/c/Users/clayc/Documents/US_Mex_InSAR') / "OPERA_data"    # WSL
mintpy_paths = list(data_storage.glob('*/mintpy'))
opera_product_paths = list(data_storage.glob('*/subset-ncs/*.nc'))

In [ ]:
import numpy as np
import h5py
from pathlib import Path
from mintpy.asc_desc2horz_vert import run_asc_desc2horz_vert
from mintpy.utils import readfile, writefile
from argparse import Namespace
import tempfile

def clip_geometry_to_displacement(geom_file, ts_file):
    """
    Clip geometry arrays in memory to match displacement spatial extent
    
    Returns:
        dict: Dictionary of clipped geometry arrays (inc_angle, azi_angle, etc.)
        dict: Attributes from timeseries file
    """
    with h5py.File(ts_file, 'r') as f:
        ts_atr = dict(f.attrs)
        ts_shape = f['timeseries'].shape[1:]  # (length, width)
    
    with h5py.File(geom_file, 'r') as f:
        geom_atr = dict(f.attrs)
        
        # Calculate pixel offsets
        ts_lat0 = float(ts_atr['Y_FIRST'])
        ts_lon0 = float(ts_atr['X_FIRST'])
        ts_lat_step = float(ts_atr['Y_STEP'])
        ts_lon_step = float(ts_atr['X_STEP'])
        
        geom_lat0 = float(geom_atr['Y_FIRST'])
        geom_lon0 = float(geom_atr['X_FIRST'])
        geom_lat_step = float(geom_atr['Y_STEP'])
        geom_lon_step = float(geom_atr['X_STEP'])
        
        # Calculate starting indices in geometry file
        y0 = int(round((ts_lat0 - geom_lat0) / geom_lat_step))
        x0 = int(round((ts_lon0 - geom_lon0) / geom_lon_step))
        
        # Clip all geometry datasets
        clipped_geom = {}
        for key in f.keys():
            clipped_geom[key] = f[key][y0:y0+ts_shape[0], x0:x0+ts_shape[1]]
    
    return clipped_geom, ts_atr

def find_common_dates(ts_files):
    """Find dates that exist in at least 2 tracks (1 asc + 1 desc)"""
    date_to_files = {}
    
    for ts_file in ts_files:
        with h5py.File(ts_file, 'r') as f:
            # Decode bytes to strings
            dates = [d.decode() if isinstance(d, bytes) else d for d in f['date'][:]]
            orbit_dir = f.attrs.get('ORBIT_DIRECTION', '').lower()
            
            for date in dates:
                if date not in date_to_files:
                    date_to_files[date] = {'asc': [], 'desc': []}
                
                direction = 'asc' if orbit_dir.startswith('asc') else 'desc'
                date_to_files[date][direction].append(ts_file)
    
    # Filter to dates with both asc and desc
    common_dates = {date: files for date, files in date_to_files.items() 
                    if files['asc'] and files['desc']}
    
    return common_dates


def extract_date_to_temp_file_with_clip(ts_file, geom_file, target_date, temp_dir):
    """Extract a single date and use clipped geometry"""
    with h5py.File(ts_file, 'r') as f:
        # Decode bytes to strings
        dates = [d.decode() if isinstance(d, bytes) else d for d in f['date'][:]]
        date_idx = np.where(np.array(dates) == target_date)[0]
        
        if len(date_idx) == 0:
            return None, None
            
        date_idx = date_idx[0]
        disp = f['timeseries'][date_idx, :, :]
        atr = dict(f.attrs)
    
    # Clip geometry in memory
    clipped_geom, _ = clip_geometry_to_displacement(geom_file, ts_file)
    
    # Create temporary files
    temp_disp = Path(temp_dir) / f"disp_{target_date}_{Path(ts_file).parent.name}.h5"
    temp_geom = Path(temp_dir) / f"geom_{target_date}_{Path(ts_file).parent.name}.h5"
    
    # Update attributes for single displacement
    atr_disp = atr.copy()
    atr_disp['FILE_TYPE'] = 'displacement'
    atr_disp['UNIT'] = 'm'
    atr_disp['DATE12'] = target_date
    
    # Write temporary displacement file
    writefile.write(disp, out_file=str(temp_disp), metadata=atr_disp)
    
    # Write temporary geometry file using MintPy's writefile
    atr_geom = atr.copy()
    atr_geom['FILE_TYPE'] = 'geometry'
    
    # Write geometry datasets
    writefile.write(
        clipped_geom, 
        out_file=str(temp_geom), 
        metadata=atr_geom
    )
    
    return str(temp_disp), str(temp_geom)

# Updated main processing function
def process_timeseries_with_mintpy(ts_files, geom_files, output_dir):
    """
    Process all timeseries using MintPy's run_asc_desc2horz_vert function
    Clips geometry in memory to match displacement dimensions
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    # Find common dates
    print("Finding common dates across tracks...")
    common_dates = find_common_dates(ts_files)
    print(f"Found {len(common_dates)} dates with both asc/desc coverage")
    
    # Create mapping of ts files to geom files
    ts_to_geom = dict(zip(ts_files, geom_files))
    
    # Process each common date
    all_dates = sorted(common_dates.keys())
    horz_stack = []
    vert_stack = []
    processed_dates = []
    ref_atr = None
    
    with tempfile.TemporaryDirectory() as temp_dir:
        for date in all_dates:
            print(f"\nProcessing date: {date}")
            
            # Get one asc and one desc file for this date
            asc_ts_file = common_dates[date]['asc'][0]
            desc_ts_file = common_dates[date]['desc'][0]
            
            # Extract to temporary files (geometry clipped in memory)
            asc_disp, asc_geom = extract_date_to_temp_file_with_clip(
                asc_ts_file, ts_to_geom[asc_ts_file], date, temp_dir
            )
            desc_disp, desc_geom = extract_date_to_temp_file_with_clip(
                desc_ts_file, ts_to_geom[desc_ts_file], date, temp_dir
            )
            
            if asc_disp is None or desc_disp is None:
                print(f"  Skipping {date} - data not found")
                continue
            
            # Set up inputs for MintPy function
            inps = Namespace()
            inps.file = [asc_disp, desc_disp]
            inps.geom_file = [asc_geom, desc_geom]
            inps.ds_name = None
            inps.horz_az_angle = 90  # East direction
            inps.one_outfile = False
            
            # Temporary output files
            temp_horz = Path(temp_dir) / f"horz_{date}.h5"
            temp_vert = Path(temp_dir) / f"vert_{date}.h5"
            inps.outfile = [str(temp_horz), str(temp_vert)]
            
            # Run MintPy decomposition
            print(f"  Running MintPy decomposition...")
            try:
                run_asc_desc2horz_vert(inps)
                
                # Read results
                horz_data, horz_atr = readfile.read(str(temp_horz))
                vert_data, vert_atr = readfile.read(str(temp_vert))
                
                horz_stack.append(horz_data)
                vert_stack.append(vert_data)
                processed_dates.append(date)
                
                if ref_atr is None:
                    ref_atr = horz_atr
                    
            except Exception as e:
                print(f"  Error processing {date}: {e}")
                continue
    
    # Save results
    print("\n" + "="*50)
    print("Saving results...")
    
    horz_stack = np.array(horz_stack)
    vert_stack = np.array(vert_stack)
    
    # Update attributes for timeseries
    ref_atr['FILE_TYPE'] = 'timeseries'
    ref_atr['UNIT'] = 'm'
    
    # Save as HDF5 timeseries
    with h5py.File(output_dir / 'horizontal_timeseries.h5', 'w') as f:
        f.create_dataset('timeseries', data=horz_stack, compression='gzip')
        f.create_dataset('date', data=np.array(processed_dates, dtype='S8'))
        for key, val in ref_atr.items():
            if isinstance(val, (str, int, float, np.integer, np.floating)):
                f.attrs[key] = val
    
    with h5py.File(output_dir / 'vertical_timeseries.h5', 'w') as f:
        f.create_dataset('timeseries', data=vert_stack, compression='gzip')
        f.create_dataset('date', data=np.array(processed_dates, dtype='S8'))
        for key, val in ref_atr.items():
            if isinstance(val, (str, int, float, np.integer, np.floating)):
                f.attrs[key] = val
    
    print(f"Saved horizontal timeseries: {output_dir / 'horizontal_timeseries.h5'}")
    print(f"Saved vertical timeseries: {output_dir / 'vertical_timeseries.h5'}")
    print(f"Total dates processed: {len(processed_dates)}")
    
    return horz_stack, vert_stack, processed_dates

In [ ]:
ts_files = list(data_storage.glob('*/*/timeseries.h5'))
geom_files = []

# filter out the problematic track without mutating the list during iteration
ts_files = [f for f in ts_files if '42264' not in str(f) and '42265' not in str(f)]       # for now, 42264 is broken, specifically the geometry file

for ts_file in ts_files:
    g_file = Path(str(ts_file).replace('timeseries', 'geometryG'))
    if g_file.exists():
        geom_files.append(g_file)

# Try decomposing with just the dlos, incidence, and azimuth angles with run_desc2horz_vert.py

In [ ]:
## First, read in files and identify which ascending and descending tracks have an overlap
# uses the get_overlap_lalo function



# Testing the run_asc_desc2horz_vert function
- problems with REF_LAT and REF_LON not being in dataset, even though they were used and identified to reformat the stack in get_opera_disp.ipynb

In [ ]:
# The below breaks because the stacked outputs do not have a REF_LAT or REF_LON.
# May need to identify these and add them into the dataset manually. Would probably entail reading the coherence data
# and selecting the coordinates where the highest coherence is at for each time step

horz, vert, dates = process_timeseries_with_mintpy(
    ts_files, 
    geom_files, 
    output_dir=data_storage / 'decomposed_timeseries'
)

In [ ]:
from argparse import Namespace
from mintpy.asc_desc2horz_vert import run_asc_desc2horz_vert

#TODO: need to somehow get the REF_LAT and REF_LON in here???

# Create input parameters
inps = Namespace()

# Required: input files (ascending and descending timeseries)
inps.file = ts_files[:2]  # List of timeseries.h5 files

# Optional: geometry files (if you want to use 2D varying incidence/azimuth angles)
inps.geom_file = geom_files[:2]  # List of geom.h5 files, same order as ts_files
# OR set to None to use constant angles from metadata
# inps.geom_file = None

# Optional: dataset name (None for timeseries, or specific dataset for other file types)
inps.ds_name = None  # For timeseries.h5 files, use None

# Optional: horizontal azimuth angle (default: -90 )
inps.horz_az_angle = -90 

# Required: output file(s)
# Option 1: Single output file containing all datasets
inps.one_outfile = True
inps.outfile = str(data_storage / 'decomposed_results.h5')

# Option 2: Separate files for horizontal and vertical
inps.one_outfile = False
inps.outfile = [
    str(data_storage / 'horizontal.h5'),
    str(data_storage / 'vertical.h5')
]

# Run the decomposition
result = run_asc_desc2horz_vert(inps)

In [ ]:
test_h5 = Path(str(ts_files[0]).replace('3065', '1092'))

PosixPath('/home/clayc/Documents/US_Mex_InSAR/OPERA_data/1092/mintpy/timeseries.h5')

In [25]:
h = 1

# test_ts = h5py.File(ts_files[h], 'r')
test_ts = h5py.File(Path(str(ts_files[0]).replace('3065', '1092')), 'r')

bperp = test_ts['bperp'][:]
dates = test_ts['date'][:]
disp = test_ts['timeseries'][:]

display(disp.shape)
display(dict(test_ts.attrs))

(18, 7943, 9567)

{'EPSG': '32614',
 'FILE_TYPE': 'timeseries',
 'HEIGHT': '750000.0',
 'LENGTH': '7943',
 'ORBIT_DIRECTION': 'Ascending',
 'PLATFORM': 'Sentinel-1',
 'UNIT': 'm',
 'UTM_ZONE': '14N',
 'WAVELENGTH': '0.05546576',
 'WIDTH': '9567',
 'X_FIRST': '352500.0',
 'X_STEP': '30.0',
 'X_UNIT': 'meters',
 'Y_FIRST': '2831190.0',
 'Y_STEP': '-30.0',
 'Y_UNIT': 'meters'}

In [35]:
test_ts

<HDF5 file "timeseries.h5" (mode r)>

In [43]:
import xarray as xr

# After running reformat_stack with reference_row/col
output_file = Path(str(ts_files[0]).replace('3065', '1092'))

# Open the file
ts_ds = xr.open_dataset(output_file)
ts_ds

<xarray.Dataset> Size: 5GB
Dimensions:     (phony_dim_0: 18, phony_dim_1: 7943, phony_dim_2: 9567)
Dimensions without coordinates: phony_dim_0, phony_dim_1, phony_dim_2
Data variables:
    bperp       (phony_dim_0) float32 72B ...
    date        (phony_dim_0) <U8 576B ...
    timeseries  (phony_dim_0, phony_dim_1, phony_dim_2) float32 5GB ...
Attributes: (12/16)
    FILE_TYPE:        timeseries
    UNIT:             m
    LENGTH:           7943
    WIDTH:            9567
    X_FIRST:          352500.0
    Y_FIRST:          2831190.0
    ...               ...
    EPSG:             32614
    UTM_ZONE:         14N
    PLATFORM:         Sentinel-1
    ORBIT_DIRECTION:  Ascending
    HEIGHT:           750000.0
    WAVELENGTH:       0.05546576

In [39]:
geom_ds = xr.open_dataset(Path('/home/clayc/Documents/US_Mex_InSAR/OPERA_data/1092/mintpy/geometryGeo.h5'))
geom_ds

<xarray.Dataset> Size: 988MB
Dimensions:         (phony_dim_0: 7943, phony_dim_1: 9567)
Dimensions without coordinates: phony_dim_0, phony_dim_1
Data variables:
    azimuthAngle    (phony_dim_0, phony_dim_1) float32 304MB ...
    height          (phony_dim_0, phony_dim_1) float32 304MB ...
    incidenceAngle  (phony_dim_0, phony_dim_1) float32 304MB ...
    shadowMask      (phony_dim_0, phony_dim_1) int8 76MB ...
Attributes: (12/16)
    FILE_TYPE:        geometry
    UNIT:             m
    LENGTH:           7943
    WIDTH:            9567
    X_FIRST:          352500.0
    Y_FIRST:          2831190.0
    ...               ...
    EPSG:             32614
    UTM_ZONE:         14N
    PLATFORM:         Sentinel-1
    ORBIT_DIRECTION:  Ascending
    HEIGHT:           750000.0
    WAVELENGTH:       0.05546576

In [46]:
coh_ds = xr.open_dataset(Path('/home/clayc/Documents/US_Mex_InSAR/OPERA_data/1092/mintpy/avgSpatialCoh.h5'))
coh_ds      #['avgSpatialCoh']

<xarray.Dataset> Size: 304MB
Dimensions:        (phony_dim_0: 7943, phony_dim_1: 9567)
Dimensions without coordinates: phony_dim_0, phony_dim_1
Data variables:
    avgSpatialCoh  (phony_dim_0, phony_dim_1) float32 304MB ...
Attributes: (12/16)
    FILE_TYPE:        mask
    UNIT:             1
    LENGTH:           7943
    WIDTH:            9567
    X_FIRST:          352500.0
    Y_FIRST:          2831190.0
    ...               ...
    EPSG:             32614
    UTM_ZONE:         14N
    PLATFORM:         Sentinel-1
    ORBIT_DIRECTION:  Ascending
    HEIGHT:           750000.0
    WAVELENGTH:       0.05546576

In [ ]:
from netCDF4 import Dataset
test_file = Dataset(nc_files[0])

temporal_coherence = test_file.variables['temporal_coherence'][:]
max_coherence = np.nanmax(temporal_coherence)
print(f"Maximum temporal coherence: {max_coherence}")

y_idx, x_idx = np.unravel_index(np.nanargmax(temporal_coherence), temporal_coherence.shape)
y_coord = test_file.variables['y'][y_idx]
x_coord = test_file.variables['x'][x_idx]
print(f"Maximum coherence location: y={y_coord}, x={x_coord} (indices: y_idx={y_idx}, x_idx={x_idx})")

In [ ]:

# Get the reference coordinates from row/col indices
ref_row, ref_col = 263, 8342
x_coord = ds.x[ref_col].values
y_coord = ds.y[ref_row].values

# Convert to lat/lon
from pyproj import Transformer
crs = ds.spatial_ref.crs_wkt
transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
ref_lon, ref_lat = transformer.transform(x_coord, y_coord)

# Add as global attributes
ds.attrs['REF_LAT'] = ref_lat
ds.attrs['REF_LON'] = ref_lon
ds.attrs['REF_Y'] = ref_row
ds.attrs['REF_X'] = ref_col

# Save back (this will overwrite)
ds.to_netcdf(output_file, engine='h5netcdf', mode='w')
ds.close()

print(f"Added REF_LAT={ref_lat:.6f}, REF_LON={ref_lon:.6f}")

/tmp/ipykernel_8019/1519690926.py:7: UserWarning: The 'phony_dims' kwarg now defaults to 'access'. Previously 'phony_dims=None' would raise an error. For full netcdf equivalence please use phony_dims='sort'.
  ds = xr.open_dataset(output_file, engine='h5netcdf')


AttributeError: 'Dataset' object has no attribute 'x'

In [ ]:
testmask = h5py.File((Path(str(ts_files[h]).replace('timeseries', 'recommended_mask_90thresh'))), 'r')
testmask['recommendedMask']

In [29]:
test_geom =  h5py.File(Path('/home/clayc/Documents/US_Mex_InSAR/OPERA_data/1092/mintpy/geometryGeo.h5'), 'r')

azi_angle = test_geom['azimuthAngle'][:]
inc_angle = test_geom['incidenceAngle'][:]
# los_east = test_geom['losEast'][:]
# los_north = test_geom['losNorth'][:]

# up = np.sqrt(1 - los_east**2 - los_north**2)

# display(los_east.shape)

In [32]:
display(dict(test_geom.attrs))

{'EPSG': '32614',
 'FILE_TYPE': 'geometry',
 'HEIGHT': '750000.0',
 'LENGTH': '7943',
 'ORBIT_DIRECTION': 'Ascending',
 'PLATFORM': 'Sentinel-1',
 'UNIT': 'm',
 'UTM_ZONE': '14N',
 'WAVELENGTH': '0.05546576',
 'WIDTH': '9567',
 'X_FIRST': '352500.0',
 'X_STEP': '30.0',
 'X_UNIT': 'meters',
 'Y_FIRST': '2831190.0',
 'Y_STEP': '-30.0',
 'Y_UNIT': 'meters'}

In [ ]:
clipped_geom, ts_atr = clip_geometry_to_displacement(geom_files[h], ts_files[h])

In [ ]:
horz, vert, dates = process_timeseries_with_mintpy(
    ts_files, 
    geom_files, 
    output_dir=data_storage / 'test_decomposition'
)

In [ ]:
import rasterio as rio
import numpy as np
from mintpy.utils.utils0 import calc_azimuth_from_east_north_obs, azimuth2heading_angle

los_east = rio.open(test.parent / 'geom_data' / 'los_east.tif').read(1).flatten()
los_north = rio.open(test.parent / 'geom_data' / 'los_north.tif').read(1).flatten()

test_az_angle = calc_azimuth_from_east_north_obs(los_east, los_north)       # azimuth angle

# Calculate up component from unit vector constraint
up = np.sqrt(1 - los_east**2 - los_north**2)

# Incidence angle is angle from vertical
test_inc_angle = np.degrees(np.arccos(up))

In [ ]:
from mintpy import asc_desc2horz_vert

asc_desc2horz_vert.asc_desc2horz_vert(dlos, los_inc_angle, los_az_angle, horz_az_angle=-90, step=20):

In [ ]:
test_ts_disp = h5py.File('/home/clayc/Documents/US_Mex_InSAR/OPERA_data/1092/mintpy/timeseries.h5', 'r')
list(test_ts_disp.keys())
# test_ts_disp['timeseries'][:]

In [ ]:
test_clsc = h5py.File(data_storage / '1092' / 'orbit_data' / 'OPERA_L2_CSLC-S1-STATIC_T005-008737-IW3_20140403_S1A_v1.0.h5', 'r')
test_clsc.keys()

#### Tentatively, I will be skipping the additional ionospheric correction that may be implemented in MintPy. The needed information for this may not be available(?) 

In [ ]:
east_tif = test.parent / 'geom_data' / 'los_east.tif'
north_tif = test.parent / 'geom_data' / 'los_north.tif'
ref_h5 = test / 'timeseries.h5'

create_geom_h5_with_ref(east_tif, north_tif, ref_h5)

In [ ]:
test_geom_h5 = h5py.File(str(ref_h5).replace('timeseries', 'geom'), 'r')
list(test_geom_h5.keys())

In [ ]:
np.unique(test_geom_h5['azimuthAngle'][:])

# Step 1 Decompose Ascending and Descending tracks into horizontal and vertical displacements from current LOS
- asc_des2horz_vert.py

In [ ]:
from mintpy import asc_desc2horz_vert
from mintpy.utils import readfile

In [ ]:
geom_files

In [ ]:
# testing the asc_des2horz_vert on the displacement dataset first
# will then test on the velocity

# TODO: check how to integrate the recommended mask 

# TODO: co-kriging between GNSS and InSAR for filling in masked/incoherent pixels as another project? May havbe to compare NN, linear, non-linear, and ML interpolation methods with co-kriging


ts_atr_list = [readfile.read_attribute(path / 'timeseries.h5') for path in mintpy_paths]
ts_density_atr_list = [readfile.read_attribute(path / 'timeseries_density.h5') for path in mintpy_paths]
vel_atr_list = [readfile.read_attribute(path / 'velocity.h5') for path in mintpy_paths]
mask_atr_list = [readfile.read_attribute(path / 'recommended_mask_90thresh.h5') for path in mintpy_paths]
coh_atr_list = [readfile.read_attribute(path / 'avgSpatialCoh.h5') for path in mintpy_paths]
geom_atr_list = [readfile.read_attribute(file) for file in geom_files]

In [ ]:
ts_atr_list

In [ ]:
geom_atr_list

In [ ]:
test_bbox = asc_desc2horz_vert.get_overlap_lalo(ts_atr_list)

In [ ]:
ts_files = [path / 'timeseries.h5' for path in mintpy_paths]

In [ ]:
asc_desc2horz_vert.run_asc_desc2horz_vert(ts_files)

# Step 2 Mosaick frames together
- image_stitch.py

In [ ]:


        # cmd = 'timeseries.h5 --yx 273 271 --figsize 8 4'
        # inps = cmd_line_parse(cmd.split())
        # obj = timeseriesViewer(inps)
        # obj.open()
        # obj.plot()

# Step 3 Save results to other data formats
- geotiffs, netcdfs, or h5s of average vertical velocity, as well as the timeseries stacks

In [ ]:
cmd = f'view.py {str(test)}/timeseries.h5'
inps = cmd_line_parse(cmd.split()[1:])
obj = viewer()
obj.configure(inps)
obj.plot()

In [ ]:
tsview([str(test / 'velocity.h5')])